In [2]:
import os
import keras
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dense , Dropout , BatchNormalization
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('dark_background')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder




In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
data = []
labels = []

for r, d, f in os.walk("/content/drive/MyDrive/Brain_Tumor_Dataset/yes"):
    for file in f:
        if file.endswith(".jpg"):
            img = Image.open(os.path.join(r, file))
            img = img.resize((128, 128))
            img = np.array(img)

            if img.shape == (128, 128, 3):
                data.append(img)
                labels.append(1)   # tumor

In [5]:
for r, d, f in os.walk("/content/drive/MyDrive/Brain_Tumor_Dataset/no"):
    for file in f:
        if file.endswith(".jpg"):
            img = Image.open(os.path.join(r, file))
            img = img.resize((128, 128))
            img = np.array(img)

            if img.shape == (128, 128, 3):
                data.append(img)
                labels.append(0)   # no tumor


In [6]:
X = np.array(data) / 255.0   # normalize
result = np.array(labels)

In [7]:
data = np.array(data)
data.shape

(139, 128, 128, 3)

In [ ]:
print(f"The total number of Images we have : {len(data)}")

# **Splitting the Data into Training and Testing**

In [ ]:
x_train , x_test , y_train , y_test = train_test_split(data , result , test_size = 0.2 , shuffle=True, random_state =0)

In [ ]:
print(f"The total number of Images we have  for Training: {len(x_train)}")
print(f"The total number of Images we have  for Testing: {len(x_test)}")

# **Build the AI Model**

In [ ]:
model = Sequential()

model.add(Conv2D(64, kernel_size=(2,2), input_shape=(128,128,3), padding = 'same'))
model.add(Conv2D(63, kernel_size =(2,2), activation = 'relu', padding = 'same'))

model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size =(2,2)))
model.add(Dropout(0.25))

model.add(Conv2D(64, kernel_size=(2,2), activation = 'relu', padding = 'same'))
model.add(Conv2D(63, kernel_size=(2,2), activation = 'relu' , padding ='same'))

model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size = (2,2), strides = (2,2)))
model.add(Dropout(0.25))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(1, activation='sigmoid'))  # FINAL ANSWER LAYER

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print(model.summary())


# **Model Evaluation**

# Testing

In [ ]:
def names(number):
  if number == 1 :
    return "It's a Tumor"
  else :
    return "It's not a Tumor"

In [ ]:
from matplotlib.pyplot import imshow
img = Image.open("/content/drive/MyDrive/Brain_Tumor_Dataset/no/1 no.jpeg")
img = img.convert("RGB")
img = img.resize((128,128))

x = np.array(img) / 255.0
x = x.reshape(1,128,128,3)
res = model.predict_on_batch(x)
prob = model.predict(x)[0][0]

if prob > 0.5:
    print(f"{prob*100:.2f}% confident: Tumor")
else:
    print(f"{(1-prob)*100:.2f}% confident: No Tumor")

imshow(img)

# **Provide a Title for your App**

In [ ]:
#@title Provide a title for your app:
heading_title = "Brain Tumor Classification app" # @param {type:"string"}

In [ ]:
#@title You can add some example images that you want to be present in your app by default. The user can see use these images to quickly and easily test the model. How many example images do you want to load?
num_examples = 2 # @param {type:"slider", min:1, max:6, step:1}

In [ ]:
#@title Enter the paths for the example images that you want displayed in your app by default. The user can use these images to quickly and easily test the model. Note: You can get the path for the file from the left sidebar. Simply run the code below, select the image file you want to include from its folder, right-click and select 'Copy path'. Then paste the path in the input box displayed:
from matplotlib import pyplot as plt
from PIL import Image

examples=[]
for i in range(num_examples):
  example_path = input(f"example_path_{i+1}:  ")
  examples.append(example_path)

#Displaying the selected images side by side
rows = 1
plt.figure(figsize=(16, 8))
for num, x in enumerate(examples):
    img = Image.open(x)
    plt.subplot(rows,6,num+1)
   # plt.title(x.split('.')[0])
    plt.axis('off')
    plt.imshow(img)


In [ ]:
#@title You can also add some description and explanation to your app's interace if you want. Go ahead and specify some text for the description and the long description (if you want to):
desc = "Brain tumor app. Let's learn!" # @param {type:"string"}
long_desc = "Select an image or upload one to predict if brain tumor is present or not" # @param {type:"string"}

In [ ]:
import gradio as gr

#@title Select a Theme for Gradio Interface:
theme_selection = "Glass" # @param ["Base", "Default", "Glass", "Monochrome", "Soft"]

theme_dict = {
    "Base": gr.themes.Base(),
    "Default": gr.themes.Default(),
    "Glass": gr.themes.Glass(),
    "Monochrome": gr.themes.Monochrome(),
    "Soft": gr.themes.Soft()
}

# The selected theme is determined by the user's dropdown selection
selected_theme = theme_dict[theme_selection]

# Now you can use the selected_theme variable when you create your Gradio interface

In [ ]:
def recognize_image(image):
    # Resize the image to the expected dimensions
    img = Image.fromarray(image).resize((128, 128))
    # Convert the image to a NumPy array
    x = np.array(img)
    # Reshape the image to match the model input
    x = x.reshape(1, 128, 128, 3)

    # Make a prediction
    prob = model.predict(x)[0][0] # Get the probability for the positive class

    if prob > 0.5:
        classification = 1 # Tumor
    else:
        classification = 0 # No Tumor

    result = f"{names(classification)}"

    return result

In [ ]:
# Assuming recognize_image, examples, heading_title, desc, long_desc, and selected_theme are defined elsewhere.

# Update the import for components
image = gr.Image()
label = gr.Label()

# Create the interface with the updated component imports
iface = gr.Interface(
    fn=recognize_image,
    inputs=image,
    outputs=label,
    examples=examples,
    title=heading_title,
    description=desc,
    article=long_desc,
    theme=selected_theme  # Make sure this is defined based on user selection as explained in previous messages
)

iface.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6f66e977f31ecaf1ca.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^